In [1]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
openai_api_key = os.environ["OPENAI_API_KEY"]

In [2]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.pydantic_v1 import BaseModel, Field

tagging_prompt = ChatPromptTemplate.from_template(
    """
Extract the desired information from the following passage.

Only extract the properties mentioned in the 'Classification' function.

Passage:
{input}
"""
)

class Classification(BaseModel):
    sentiment: str = Field(description="The sentiment of the text")
    political_tendency: str = Field(
        description="The political tendency of the user"
    )
    language: str = Field(description="The language the text is written in")


# LLM
llm = ChatOpenAI(temperature=0, model="gpt-4o-mini").with_structured_output(
    Classification
)

tagging_chain = tagging_prompt | llm

In [4]:
Comment1 = "Başkan Trump'ın liderliğinin ve geçmişteki başarılarının Amerikalılarla yeniden yankı bulacağından eminim. Ekonomik büyüme ve ulusal güvenlik konusundaki kararlı duruşu, ülkemizin tam da bu kritik dönemde ihtiyaç duyduğu şey. Amerika'yı yeniden büyük yapabilecek kanıtlanmış liderliği geri getirmeliyiz!"

In [5]:
tagging_chain.invoke({"input": Comment1})

Classification(sentiment='positive', political_tendency='supportive', language='Turkish')

In [6]:
Comment2 = "Başkan Biden'ın şefkatli ve istikrarlı yaklaşımının şu anda ülkemiz için hayati olduğuna inanıyorum. Sağlık reformuna, iklim değişikliğiyle mücadeleye ve uluslararası ittifaklarımızı yeniden güçlendirmeye olan bağlılığı son derece önemli. İlerlemeye devam etme ve tüm Amerikalılar için faydalı bir gelecek sağlama zamanı geldi."

In [7]:
tagging_chain.invoke({"input": Comment2})

Classification(sentiment='şefkatli ve istikrarlı', political_tendency='destekleyici', language='Türkçe')

In [9]:
#enum kullanalım

In [10]:
class Classification(BaseModel):
    sentiment: str = Field(..., enum=["mutlu", "nötr", "üzgün"])
    political_tendency: str = Field(
        ...,
        description="The political tendency of the user",
        enum=["muhafazakâr", "liberal", "bağımsız"],
    )
    language: str = Field(
        ..., enum=["Türkçe", "İngilizce"]
    )

In [11]:
tagging_prompt = ChatPromptTemplate.from_template(
    """
Extract the desired information from the following passage.

Only extract the properties mentioned in the 'Classification' function.

Passage:
{input}
"""
)

llm = ChatOpenAI(temperature=0, model="gpt-4o-mini").with_structured_output(
    Classification
)

tagging_chain = tagging_prompt | llm

In [12]:
tagging_chain.invoke({"input": Comment1})

Classification(sentiment='mutlu', political_tendency='muhafazakâr', language='Türkçe')

In [13]:
tagging_chain.invoke({"input": Comment2})

Classification(sentiment='mutlu', political_tendency='liberal', language='Türkçe')